# Phase 1 — Train text-aware DTD (loss + OCR prior)

Fine-tunes the DocTamper **DTD** model with our **`CombinedTamperLoss`** (CE + Dice + boundary), and
documents how to inject the **OCR text prior** (`TextPriorFusion`) into the decoder.


In [ ]:
# --- Setup: clone repos + install a CONFLICT-FREE stack (modern torch is fine) ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys
%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone -q https://github.com/qcf-568/DocTamper.git
if not os.path.exists('/content/HLCV-Project'):
    !git clone -q https://github.com/SamiraAbedini/HLCV-Project.git

# Libs the code needs. Do NOT pin torch (use Colab's). Do NOT pin efficientnet_pytorch:
# smp 0.2.1 requires efficientnet 0.6.3, and pinning 0.7.1 makes pip abort the whole install.
!pip -q install lmdb six albumentations timm==0.4.12 segmentation_models_pytorch==0.2.1 easyocr

# Correct jpegio = dwgoon's (provides jpegio.read used by DocTamper). The PyPI 'jpegio'
# lacks .read, so build from source. Installed BEFORE anything imports jpegio (no restart).
!pip -q uninstall -y jpegio
!apt-get -qq install -y libjpeg-dev > /dev/null
!rm -rf /content/jpegio && git clone -q https://github.com/dwgoon/jpegio.git /content/jpegio
%cd /content/jpegio
!pip -q install .
%cd /content/DocTamper/models

sys.path.append('/content/HLCV-Project')   # our src/ modules
import jpegio
print('jpegio.read available:', hasattr(jpegio, 'read'))
print('setup done. If jpegio.read is False, do Runtime > Restart and re-run this cell once.')

In [ ]:
# --- Stage data + pickles + backbones into the models/ working dir ---
# TamperDataset opens `lmdb.open(roots)` and reads `qt_table.pk` and `pks/{roots}_{minq}.pk`
# RELATIVE to the current dir, so everything must live in /content/DocTamper/models.
import os, glob, getpass
%cd /content/DocTamper/models

ROOTS = 'DocTamperV1-FCD'   # lmdb folder name == key used by pks/{ROOTS}_{minq}.pk
MINQ  = 75                  # must match an existing pks/{ROOTS}_{MINQ}.pk file

# 1) qt_table.pk and pks/ ship in the repo root.
if not os.path.exists('qt_table.pk'):
    !cp ../qt_table.pk ./
if not os.path.exists('pks'):
    !cp -r ../pks ./

# 2) LMDB dataset: unzip your archive here (password typed at runtime, not stored).
ZIP_PATH = '/content/drive/MyDrive/DocTamperV1-FCD.zip'
if not os.path.exists(f'{ROOTS}/data.mdb'):
    found = glob.glob(f'/content/**/{ROOTS}/data.mdb', recursive=True)
    if found:
        !ln -s "{os.path.dirname(found[0])}" "./{ROOTS}"
    else:
        assert os.path.exists(ZIP_PATH), f'Dataset zip not found: {ZIP_PATH}'
        pw = getpass.getpass('DocTamper zip password: ')
        !unzip -P "$pw" "{ZIP_PATH}" -d ./
        del pw
assert os.path.exists(f'{ROOTS}/data.mdb'), f'Expected {ROOTS}/data.mdb in models/ dir.'

# 3) Checkpoints (seg_dtd loads the backbones at construction). Download the author's folder:
#    https://drive.google.com/drive/folders/11Ep8PJIrlIveudQaRulDOBENHGqw762a
#    into MyDrive/checkpoints -> vph_imagenet.pt, swin_imagenet.pt, dtd_doctamper.pth
CKPT_DIR = '/content/drive/MyDrive/checkpoints'
for f in ['vph_imagenet.pt', 'swin_imagenet.pt', 'dtd_doctamper.pth']:
    if not os.path.exists(f) and os.path.exists(os.path.join(CKPT_DIR, f)):
        !cp "{CKPT_DIR}/{f}" ./
for f in ['vph_imagenet.pt', 'swin_imagenet.pt']:
    assert os.path.exists(f), (
        f'{f} not found. Download the checkpoints folder (link above) into {CKPT_DIR} -- '
        'seg_dtd loads these backbones when the model is built.')
print('staging done. backbones present; pks sample:', sorted(os.listdir('pks'))[:5], '...')

In [ ]:
# --- Load TamperDataset from eval_dtd.py WITHOUT running its argparse/eval code ---
# eval_dtd.py is a script (top-level argparse + eval loop), so a plain import runs it and
# crashes on Colab's argv. We exec ONLY its imports + the dataset/metric classes.
import ast, torch
from torch.utils.data import DataLoader
%cd /content/DocTamper/models

src = open('eval_dtd.py').read()
tree = ast.parse(src)
keep = [n for n in tree.body
        if isinstance(n, (ast.Import, ast.ImportFrom))
        or (isinstance(n, ast.ClassDef) and n.name in ('TamperDataset', 'IOUMetric'))]
ns = {}
exec(compile(ast.Module(body=keep, type_ignores=[]), 'eval_dtd_extract', 'exec'), ns)
TamperDataset = ns['TamperDataset']
print('TamperDataset loaded (script body not executed)')

train_set = TamperDataset(ROOTS, mode='train', minq=MINQ)
# num_workers=0 surfaces errors directly (worker errors get wrapped); raise it once it runs.
train_loader = DataLoader(train_set, batch_size=4, shuffle=True, num_workers=0, drop_last=True)

batch = next(iter(train_loader))
print('batch keys:', list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'  {k:6s} {tuple(v.shape)} {v.dtype}')
# Expected: image (B,3,H,W) float | label (B,1,H,W) long {0,1} | rgb=DCT | q=quant table | i=quality

In [ ]:
# --- Model + our combined loss + optimizer ---
import torch
from torch.cuda.amp import autocast, GradScaler

# Modern torch (>=2.6) defaults to torch.load(weights_only=True), but the author's backbones
# and checkpoint are FULL pickled objects -> force weights_only=False so the internal loads in
# dtd.py succeed. UNTRUSTED load: acceptable ONLY here (disposable session, author's files).
_ORIG_LOAD = torch.load
torch.load = lambda *a, **k: _ORIG_LOAD(*a, **{**k, 'weights_only': False})

from dtd import seg_dtd                 # DocTamper model (imports fine on modern torch)
from src.losses import CombinedTamperLoss

device = 'cuda'
model = seg_dtd('', 2).to(device)      # loads vph_imagenet.pt / swin_imagenet.pt from cwd

# Optional: fine-tune from the released checkpoint instead of training from scratch.
FINETUNE_PTH = 'dtd_doctamper.pth'     # staged into models/ by the previous cell; None to skip
if FINETUNE_PTH and os.path.exists(FINETUNE_PTH):
    sd = torch.load(FINETUNE_PTH, map_location='cpu')['state_dict']
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print('loaded checkpoint | missing:', len(missing), 'unexpected:', len(unexpected))

criterion = CombinedTamperLoss(lambda_dice=1.0, lambda_bound=0.5)  # ignore_index=100 (harmless on {0,1})
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = GradScaler()
print('model + loss ready')

In [ ]:
# --- Training loop (loss ablation: our CombinedTamperLoss) ---
import torch.nn.functional as F

def forward_dtd(model, batch):
    """VALIDATE against eval_dtd.py's inference call `model(data, dct, qs)`.
    Check the printed batch shapes; if the model rejects an arg, copy eval_dtd.py's
    exact tensor prep (dtypes / unsqueeze) here."""
    data = batch['image'].to(device).float()
    dct  = batch['rgb'].to(device).float()
    qs   = batch['q'].to(device)
    return model(data, dct, qs)         # expected logits (B, 2, H, W)

EPOCHS, MAX_STEPS = 1, 200              # keep small for a first validation run
model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        if step >= MAX_STEPS:
            break
        target = batch['label'].squeeze(1).long().to(device)   # (B, H, W)
        optimizer.zero_grad()
        with autocast():
            logits = forward_dtd(model, batch)
            if logits.shape[-2:] != target.shape[-2:]:           # align if head outputs H/2
                logits = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False)
            loss, parts = criterion(logits, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if step % 20 == 0:
            print(f'e{epoch} s{step:04d} | total {parts["total"]:.4f} '
                  f'ce {parts["ce"]:.4f} dice {parts["dice"]:.4f} bound {parts["boundary"]:.4f}')
print('training step finished')

## Adding the OCR text prior (`TextPriorFusion`) — requires a small `dtd.py` edit

The fusion `F̂_l = φ_l([F_l, M_l])` operates on an **internal decoder feature**, so it cannot be bolted on
from outside the model. The minimal change:

1. **Pass `M_text` into `forward`.** Build the binary text mask per image with `src.text_prior` (cache it
   per LMDB index so EasyOCR runs once), add it to the batch, and extend
   `seg_dtd.forward(self, x, dct, qt, text_mask=None)` to thread it through to the decoder.
2. **Insert the module in the decoder.** In `dtd.py`'s `MID` decoder (around the `FU` module that fuses
   visual/frequency features), add `self.fusion = TextPriorFusion(in_channels=C)` for that stage's
   channel count `C`, and call `feat = self.fusion(feat, text_mask)` before the remaining decoder layers.
3. **Phase-2 (TA's note):** instead of an external EasyOCR mask, add a lightweight OCR/text head on the
   Visual Perception Head features and use *its* output as `M_text` — one backbone, two heads. Select
   that head by its **recall** with `OCRCoverageMeter` (the other notebook).

Until step 2 is wired, run this notebook as the **loss ablation**. See
[`docs/INTEGRATION.md`](../docs/INTEGRATION.md) for the full plan.

**Reminder:** never commit the dataset, `*.pth`, or `*.pk` — `.gitignore` blocks them; keep them in the
Colab session / Drive only.